# Analyze one dataset with NEURAL

Run NN ROI detection + PeakCaller event detection on **one recording or a folder of recordings**, then render trace plots, ROI overlays, and a summary panel.

This notebook handles **one dataset**. For looping over multiple experiments/datasets, see `Paper/run_all_analyses.py`.

## 1. Pick a config

Each `NEURAL/config_examples/*.json` is a template for one of the four NN models (iglusnfr / calcium_imaging / igabasnfr / human_gc8). Copy one and edit, or build a dict inline below.

**Most-edited fields, in order of likelihood:**
1. `peak_detection.threshold` – per-experiment tuning knob
2. `nn.model` – which of the 4 production NNs to use
3. `visualization.kind` and `n_rois_per_video`
4. `input.path` – your recording or folder
5. `input.fps` / `pixel_size_um` – only if your video is `.tif` or the nd2 metadata is wrong (set to a number to override)

In [ ]:
import sys, json
from pathlib import Path

# Make NEURAL importable when running from inside the NEURAL/ folder
sys.path.insert(0, str(Path.cwd().parent))

config = {
    'peak_detection': {
        'threshold': 0.03,         # <-- the knob you'll tune most
        'baseline':  'convex',     # 'convex' (GCaMP/iGluSnFr) or 'als' (iGABASnFr)
        'negate':    False         # True for negative-going indicators (iGABASnFr)
    },
    'nn': {
        'model':                  'iglusnfr',  # 'iglusnfr' | 'calcium_imaging' | 'igabasnfr' | 'human_gc8'
        'min_roi_px':             32,
        'fg_threshold':           0.7,
        'use_cached_inference':   True         # False to re-run NN even if a cached output exists
    },
    'visualization': {
        'kind':              'all',            # 'trace_grid' | 'roi_overlay' | 'summary_panel' | 'all'
        'n_rois_per_video':  10,
        'roi_selection':     'top_activity',   # 'top_activity' | 'random' | 'all' | 'by_index'
        'roi_indices':       None,
        'videos_to_show':    None              # None = all in input folder
    },
    'input': {
        'path':          'C:/path/to/one_recording.nd2',   # <-- EDIT ME (file or folder)
        'format':        'auto',
        'inference_dir': None,                 # path to a cached NN output folder (optional)
        'fps':           None,                 # None = read from nd2 metadata; number = manual override
        'pixel_size_um': None,                 # None = read from nd2; number = override (e.g. 0.5909)
        'n_frames':      None,                 # None = use all; number = truncate
        'duration_s':    None
    },
    'output': {
        'dir':    './output_analysis',
        'format': 'pdf',
        'dpi':    140
    }
}
print(json.dumps(config, indent=2))

## 2. Run

In [ ]:
from NEURAL.figure_engine import render_dataset
results = render_dataset(config)
results

## 3. Inspect the outputs

Per-recording PDFs landed in `output.dir`. Open them, or programmatically:

In [ ]:
from pathlib import Path
for stem, kinds in results.items():
    print(stem)
    for k, p in kinds.items():
        print(f'  {k:>14s}: {p}')